# Sesión 2 · Calidad de datos y valores faltantes

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete
**Martes 20 de octubre de 2026 · 6:30 a 8:30 pm (CDMX)** · Módulo I: *Datos y su calidad*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

## 🎯 Objetivo de la sesión
**Auditar la calidad de un dataset y elegir de forma justificada cómo tratar los faltantes.**

Al terminar podrás:
1. Aplicar una lista de verificación de calidad a cualquier dataset.
2. Detectar y resolver duplicados exactos y por llave.
3. Corregir inconsistencias de texto, categorías mal escritas, rangos imposibles y unidades.
4. Distinguir los mecanismos de faltantes (MCAR, MAR, MNAR) y perfilarlos.
5. Comparar eliminación, imputación simple, imputación por grupo e indicador de faltante, y medir su efecto en las distribuciones.

## 🗺️ Agenda (120 min)
| Min | Bloque |
|---|---|
| 10 | Repaso y lista de verificación de calidad |
| 20 | 1. Duplicados: exactos y por llave |
| 25 | 2. Inconsistencias: texto, categorías, rangos imposibles, unidades |
| 15 | 3. Mecanismos de faltantes y cómo perfilarlos |
| 30 | 4. Tratamiento de faltantes y su efecto en las distribuciones |
| 20 | 5. Ejercicio (`titanic` + `penguins`) y cierre |

🟢 = ejercicio básico · 🔴 = reto opcional

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "penguins.csv").exists())
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)

pinguinos = pd.read_csv(DATA / "penguins.csv")
titanic = pd.read_csv(DATA / "titanic.csv")
print("penguins:", pinguinos.shape, "| titanic:", titanic.shape)

---
## 0. Repaso y lista de verificación de calidad (10 min)

Ayer construimos el diccionario de datos y vimos que `penguins` tiene faltantes. Hoy los auditamos con orden. **Lista de verificación** que usaremos en todo el curso (y en el proyecto):

| # | Pregunta | Herramientas |
|---|---|---|
| 1 | ¿Forma y tipos correctos? | `shape`, `info`, `dtypes` |
| 2 | ¿Hay una llave? ¿Es única? | `duplicated(subset=...)`, `nunique` |
| 3 | ¿Filas duplicadas exactas? | `duplicated()`, `drop_duplicates()` |
| 4 | ¿Categorías consistentes? | `value_counts(dropna=False)`, `str.strip/lower` |
| 5 | ¿Valores en rangos posibles? ¿Unidades homogéneas? | `describe`, `between`, reglas de dominio |
| 6 | ¿Cuántos faltantes, dónde y por qué? | `isna().sum()`, patrones por grupo |
| 7 | ¿Qué decisión tomo y cómo afecta? | comparar antes/después |

> 💡 **Regla de oro:** nunca limpies "en silencio". Cada decisión se **documenta** y se **cuantifica** (cuántas filas afecta y cómo cambia el resultado).

### 🟢 Ejercicio 1 · Perfil rápido reutilizable
Escribe una función `perfil_rapido(df)` que devuelva un DataFrame con una fila por columna y las columnas `dtype`, `n_faltantes`, `pct_faltantes` (0–100, con 1 decimal), `n_unicos` y `ejemplo` (el primer valor no nulo). Pruébala con `pinguinos` y con `titanic`. Te servirá en el resto del curso.

In [ ]:
def perfil_rapido(df):
    # TU CÓDIGO AQUÍ
    return None

perfil_rapido(pinguinos)

---
## 1. Duplicados: exactos y por llave (20 min)

- **Duplicado exacto:** toda la fila se repite. Suele deberse a cargas repetidas o *joins* mal hechos.
- **Duplicado por llave:** la llave (p. ej., `id`) se repite aunque otras columnas difieran. Es más peligroso: hay que **decidir qué registro es el bueno**.

`df.duplicated()` marca con `True` cada fila que **ya apareció antes** (`keep="first"`); con `keep=False` marca todas las copias.

In [ ]:
# Un pequeño registro de inscripciones al curso
inscripciones = pd.DataFrame({
    "id_alumno": [101, 102, 103, 102, 104, 105, 105],
    "nombre":    ["Ana", "Luis", "Marta", "Luis", "Iván", "Sofía", "Sofía"],
    "correo":    ["ana@x.mx", "luis@x.mx", "marta@x.mx", "luis@x.mx", "ivan@x.mx", "sofia@x.mx", "sofia.r@x.mx"],
    "fecha":     ["2026-10-01", "2026-10-02", "2026-10-02", "2026-10-02", "2026-10-05", "2026-10-06", "2026-10-09"],
})
print("duplicados exactos:", inscripciones.duplicated().sum())
print("duplicados por llave:", inscripciones.duplicated(subset=["id_alumno"]).sum())
inscripciones[inscripciones.duplicated(subset=["id_alumno"], keep=False)]

In [ ]:
# Política explícita: para cada alumno conservamos el registro MÁS RECIENTE (correo actualizado)
inscripciones_limpio = (
    inscripciones
    .sort_values("fecha")
    .drop_duplicates(subset=["id_alumno"], keep="last")
    .sort_values("id_alumno")
)
inscripciones_limpio

### ⚠️ ¿Todo duplicado exacto es un error? El caso `titanic`

In [ ]:
n_dup = titanic.duplicated().sum()
print(f"titanic tiene {n_dup} filas idénticas a otra anterior ({n_dup / len(titanic):.1%})")
titanic[titanic.duplicated(keep=False)].sort_values(["class", "sex", "age"]).head(6)

`titanic` **no tiene llave** (no hay nombre ni número de boleto). Dos hombres de tercera clase, sin edad registrada, que embarcaron en Southampton y pagaron la misma tarifa son **personas distintas** con los mismos atributos. Borrar esas filas eliminaría pasajeros reales y sesgaría el análisis.

> Sin una llave, un duplicado exacto es solo **sospechoso**, no necesariamente un error. Decide con conocimiento del origen de los datos.

### Un dataset "sucio" para practicar
La función de abajo toma `penguins` y le introduce, con semilla fija, problemas típicos del mundo real: una llave `id_muestra`, duplicados, texto inconsistente, unidades mezcladas y códigos imposibles. **No necesitas estudiar la función**: tu trabajo es *descubrir* los problemas como si no supieras qué tiene.

In [ ]:
def ensuciar_pinguinos(df, seed=401):
    # Devuelve una copia de penguins con problemas de calidad típicos (reproducibles)
    r = np.random.default_rng(seed)
    d = df.copy().reset_index(drop=True)
    d.insert(0, "id_muestra", [f"P{i:04d}" for i in range(1, len(d) + 1)])
    con_medidas = d.index[d["body_mass_g"].notna()]

    # Texto inconsistente en especie, isla y sexo
    variantes = {"Adelie": ["adelie", " Adelie", "ADELIE", "Adelie "],
                 "Gentoo": ["gentoo", "Gentoo ", "Gentoo penguin"],
                 "Chinstrap": ["chinstrap", "Chinstrap ", "Chinstarp"]}
    for i in r.choice(len(d), 30, replace=False):
        d.loc[i, "species"] = r.choice(variantes[d.loc[i, "species"]])
    i = r.choice(len(d), 20, replace=False)
    d.loc[i, "island"] = d.loc[i, "island"].str.lower() + " "
    i = r.choice(d.index[d["sex"].notna()], 25, replace=False)
    d.loc[i, "sex"] = d.loc[i, "sex"].map(lambda s: r.choice({"MALE": ["M", "male", "Male"], "FEMALE": ["F", "female", "Female"]}[s]))
    d.loc[r.choice(d.index[d["sex"].isna()], 3, replace=False), "sex"] = "."

    # Índices disjuntos para cada tipo de problema numérico
    perm = r.permutation(con_medidas)
    i_kg, i_aleta, i_pico, i_recaptura = perm[:8], perm[8:12], perm[12:14], perm[14:18]
    # Unidades mezcladas: algunas masas capturadas en kg
    d.loc[i_kg, "body_mass_g"] = d.loc[i_kg, "body_mass_g"] / 1000
    # Códigos y valores imposibles
    d.loc[i_aleta, "flipper_length_mm"] = [-999, -999, 0, 2100]
    d.loc[i_pico, "bill_length_mm"] = 999

    # Duplicados exactos (carga repetida) y por llave (re-captura con otra masa)
    exactos = d.sample(6, random_state=seed)
    recaptura = d.loc[i_recaptura].copy()
    recaptura["body_mass_g"] = recaptura["body_mass_g"] + 75
    return pd.concat([d, exactos, recaptura]).sample(frac=1, random_state=seed).reset_index(drop=True)

sucio = ensuciar_pinguinos(pinguinos)
print(sucio.shape)
sucio.head()

### 🟢 Ejercicio 2 · Duplicados en `sucio`
1. ¿Cuántos duplicados **exactos** hay? Elimínalos y guarda el resultado en `sin_dup`.
2. En `sin_dup`, ¿cuántos `id_muestra` siguen **repetidos**? Muestra todas las filas de esos ids, ordenadas por `id_muestra`. ¿En qué columna difieren?

In [ ]:
# TU CÓDIGO AQUÍ
sin_dup = None

### 🔴 Reto 1 · Resolver duplicados por llave con una regla
Escribe una función `columnas_en_conflicto(df, llave)` que, para cada valor de la llave repetido, liste **qué columnas** tienen valores distintos. Luego aplica una política: conservar la **primera aparición en el archivo** de cada `id_muestra` y guarda el resultado en `sin_dup_llave`. ¿Qué otra política sería razonable?

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---
## 2. Inconsistencias (25 min)

### 2.1 Texto y categorías
El primer síntoma aparece con `value_counts(dropna=False)`: la misma categoría escrita de varias formas.

In [ ]:
for col in ["species", "island", "sex"]:
    print(sucio[col].value_counts(dropna=False), "\n")

In [ ]:
# Paso 1: normalizar espacios y mayúsculas
especie_norm = sucio["species"].str.strip().str.lower()
print(especie_norm.value_counts())

# Paso 2: mapear variantes a la categoría canónica (lo que no esté en el mapa queda como NaN -> ¡revisar!)
mapa_especie = {"adelie": "Adelie", "gentoo": "Gentoo", "gentoo penguin": "Gentoo",
                "chinstrap": "Chinstrap", "chinstarp": "Chinstrap"}
especie_limpia = especie_norm.map(mapa_especie)
print("\nsin mapear:", especie_limpia.isna().sum())
especie_limpia.value_counts()

### 2.2 Rangos imposibles, códigos centinela y unidades
`describe()` y un poco de conocimiento del dominio delatan los problemas: una aleta no mide −999 mm ni 2.1 m, y un pingüino no pesa 4.2 g.

In [ ]:
sucio.describe().T

In [ ]:
# Valores sospechosamente pequeños en la masa: ¿se capturaron en kg?
masa = sucio["body_mass_g"]
print(masa[masa < 100].sort_values().to_list())

# Histograma en escala log para ver los dos "grupos" de unidades
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(np.log10(masa.dropna()), bins=40, color="steelblue")
ax.set(xlabel="log10(body_mass_g)", ylabel="frecuencia", title="Masa: dos escalas mezcladas (kg y g)")
plt.show()

**Criterios típicos** (y siempre documentados):
- **Códigos centinela** (`-999`, `0`, `9999`, `"."`, `"NA"`): convertirlos a `NaN`.
- **Unidades:** si la regla es inequívoca (masa < 100 ⇒ estaba en kg), convertir; si no, marcar y consultar.
- **Fuera de rango plausible:** a `NaN` si es imposible; si solo es raro, **no** se toca aquí (eso es un *outlier*, sesión 3).

### 🟢 Ejercicio 3 · Normaliza las categorías
A partir de `sin_dup_llave` (si no hiciste el reto, usa `sin_dup.drop_duplicates(subset=["id_muestra"])`), crea `limpio` con `species`, `island` y `sex` normalizadas a: `Adelie/Chinstrap/Gentoo`, `Biscoe/Dream/Torgersen` y `Male/Female` (con `"."` convertido a `NaN`). Verifica con `value_counts(dropna=False)` que no queden variantes.

In [ ]:
base_limpieza = sin_dup.drop_duplicates(subset=["id_muestra"]) if sin_dup is not None else None
# TU CÓDIGO AQUÍ
limpio = None

### 🟢 Ejercicio 4 · Rangos y unidades
En `limpio`:
1. Convierte a gramos las masas que están en kg (valores < 100).
2. Pon en `NaN` las aletas fuera de 150–250 mm y los picos fuera de 30–65 mm.
3. Reporta **cuántos valores** cambió cada regla. Compara `describe()` antes y después.

In [ ]:
# TU CÓDIGO AQUÍ

### 🔴 Reto 2 · Validador de reglas
Escribe `validar(df, reglas)` donde `reglas` es un diccionario `{columna: (mínimo, máximo)}` y devuelve un DataFrame con, por columna: `n_fuera_de_rango`, `pct_fuera_de_rango` y una lista de hasta 5 ejemplos. Aplícalo a `sucio` y a `limpio`. (Este tipo de función es oro en el proyecto integrador.)

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---
## 3. Mecanismos de faltantes y cómo perfilarlos (15 min)

| Mecanismo | La probabilidad de faltar depende de… | Ejemplo | Consecuencia |
|---|---|---|---|
| **MCAR** (*Missing Completely At Random*) | nada: azar puro | Se cae una hoja del cuaderno de campo | Eliminar filas solo reduce *n*; no sesga |
| **MAR** (*Missing At Random*) | **otras variables observadas** | Falta la edad más en 3ra clase | Sesga si se ignora; se corrige condicionando (imputar por grupo) |
| **MNAR** (*Missing Not At Random*) | **el propio valor faltante** | Los de ingreso alto no reportan su ingreso | Sesga y los datos solos no pueden demostrarlo ni corregirlo |

⚠️ No existe una prueba que confirme MCAR o MAR con certeza: con los datos solo podemos **descartar** MCAR (si la falta depende de algo observado). MNAR se argumenta con conocimiento del dominio.

### Simulación: el mismo dataset con tres mecanismos
Tomamos las masas completas de `penguins`, les quitamos ~30% de valores con cada mecanismo y comparamos la media observada con la verdadera.

In [ ]:
completos = pinguinos.dropna(subset=["body_mass_g"]).reset_index(drop=True)
masa_real = completos["body_mass_g"]

p_mcar = np.full(len(completos), 0.30)                                       # igual para todos
p_mar = np.where(completos["species"] == "Gentoo", 0.60, 0.15)               # depende de la especie (observada)
p_mnar = 0.6 / (1 + np.exp(-(masa_real - 4500) / 300))                       # depende de la propia masa

sim = completos[["species", "body_mass_g"]].copy()
for nombre, p in {"MCAR": p_mcar, "MAR": p_mar, "MNAR": p_mnar}.items():
    falta = rng.random(len(sim)) < p
    sim[f"masa_{nombre}"] = masa_real.mask(falta)

resumen_sim = pd.DataFrame({
    "% faltante": sim.filter(like="masa_").isna().mean() * 100,
    "media observada": sim.filter(like="masa_").mean(),
}).round(1)
resumen_sim["sesgo vs. real"] = (resumen_sim["media observada"] - masa_real.mean()).round(1)
print(f"Media real: {masa_real.mean():.1f} g")
resumen_sim

In [ ]:
# Perfil por grupo: ¿la falta depende de la especie?
sim.filter(like="masa_").isna().groupby(sim["species"]).mean().round(2)

- **MCAR:** la media casi no cambia y la proporción de faltantes es pareja entre especies.
- **MAR:** la falta es mucho mayor en Gentoo (¡lo vemos en el perfil!) y la media observada **subestima** porque perdemos a los más pesados. Como la causa (especie) está observada, podemos corregir condicionando en ella.
- **MNAR:** también subestima, pero el perfil por especie **no** revela la causa completa: depende del valor que no vemos.

### Herramientas para perfilar faltantes

In [ ]:
# 1) Conteo y porcentaje por columna
faltantes = pd.DataFrame({"n": pinguinos.isna().sum(), "%": (pinguinos.isna().mean() * 100).round(1)})
print(faltantes, "\n")

# 2) Patrones: qué combinaciones de columnas faltan juntas
patrones = pinguinos.isna().value_counts().rename("n_filas")
print(patrones, "\n")

# 3) Mapa visual de faltantes (cada línea clara es un NaN)
fig, ax = plt.subplots(figsize=(7, 3))
sns.heatmap(pinguinos.isna().T, cbar=False, cmap="Greys", ax=ax)
ax.set(xlabel="fila", title="Faltantes en penguins")
ax.set_xticks([])
plt.show()

### 🟢 Ejercicio 5 · ¿MCAR, MAR o MNAR?
Clasifica cada caso y justifica en una frase:
1. En una encuesta de salud, las personas con mayor consumo de alcohol tienden a no contestar la pregunta sobre consumo de alcohol.
2. Un lector de código de barras falla al azar en 2% de las lecturas.
3. En `tips`, imagina que el mesero olvidó anotar la propina más a menudo en las cenas de fin de semana (cuando hay más trabajo), y `day` y `time` sí están registrados.
4. En `penguins`, a 2 pingüinos les faltan **todas** las medidas.

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 4. Tratamiento de faltantes y su efecto (30 min)

| Estrategia | Cuándo | Costo |
|---|---|---|
| **Eliminar filas** (`dropna`) | Pocos faltantes y MCAR | Pierde *n*; sesga si no es MCAR |
| **Eliminar columna** | La columna casi no tiene datos o no es relevante | Pierde la variable |
| **Imputar media/mediana global** | Rápido, pocos faltantes | **Reduce la varianza**, crea un pico artificial, distorsiona relaciones |
| **Imputar por grupo** (`groupby().transform`) | MAR con la causa observada | Mejor, pero sigue subestimando la variabilidad dentro del grupo |
| **Indicador de faltante** | Siempre que se impute: conserva la información de *que faltaba* | Una columna más |
| (Avanzado) Imputación múltiple / modelos | Análisis inferencial serio | Más complejo; fuera del alcance de hoy |

Usemos el escenario **MAR** de la simulación, donde conocemos la verdad.

In [ ]:
mar = sim[["species", "body_mass_g", "masa_MAR"]].copy()

mar["elim"] = mar["masa_MAR"]                                                  # dropna implícito al resumir
mar["imp_media"] = mar["masa_MAR"].fillna(mar["masa_MAR"].mean())
mar["imp_mediana"] = mar["masa_MAR"].fillna(mar["masa_MAR"].median())
mar["imp_grupo"] = mar["masa_MAR"].fillna(mar.groupby("species")["masa_MAR"].transform("median"))
mar["faltaba"] = mar["masa_MAR"].isna()                                        # indicador de faltante

estrategias = ["body_mass_g", "elim", "imp_media", "imp_mediana", "imp_grupo"]
tabla = mar[estrategias].agg(["count", "mean", "median", "std"]).T.round(1)
tabla.index = ["real (sin faltantes)", "eliminar", "media global", "mediana global", "mediana por especie"]
tabla

In [ ]:
# Efecto visual en la distribución
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharex=True, sharey=True)
for ax, col, titulo in zip(axes, ["elim", "imp_media", "imp_grupo"],
                           ["Eliminar", "Imputar media global", "Imputar mediana por especie"]):
    sns.histplot(mar["body_mass_g"], bins=30, color="lightgray", label="real", ax=ax)
    sns.histplot(mar[col].dropna(), bins=30, color="tab:blue", alpha=0.6, label=titulo, ax=ax)
    ax.set(title=titulo, xlabel="masa corporal (g)")
axes[0].legend()
plt.tight_layout()
plt.show()

**Lecturas clave:**
- **Eliminar** con MAR sesga la media hacia abajo (perdimos Gentoo pesados).
- **Media/mediana global** crea un **pico artificial** y **encoge la desviación estándar**: subestima la variabilidad y hará que, más adelante, los intervalos de confianza sean falsamente estrechos.
- **Por grupo** recupera mejor la media y la forma, porque condiciona en la variable que explica la falta.
- Siempre conviene guardar el **indicador** (`faltaba`) para poder revisar después si los imputados se comportan distinto.

---
## 5. Ejercicio de la sesión (20 min + práctica)

### Parte A · `titanic`

### 🟢 Ejercicio 6 · Cuantifica los faltantes
Calcula el número y el porcentaje de faltantes de `age`, `deck` y `embark_town`. ¿Qué harías con cada una a primera vista?

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 7 · ¿La falta de `age` depende de `class`?
Calcula la proporción de `age` faltante por `class` (y, si quieres, por `sex` y por `embark_town`). Con base en eso, ¿es plausible que sea MCAR? Haz una gráfica de barras de la proporción por clase.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 8 · Mediana global vs. mediana por `class` y `sex`
1. Crea `age_med_global` (imputando con la mediana global) y `age_med_grupo` (con la mediana por `class` y `sex`, usando `groupby(...).transform("median")`), además de un indicador `age_faltaba`.
2. Compara media, mediana y desviación estándar de `age` original y de las dos versiones imputadas.
3. Grafica las tres distribuciones (histograma o KDE). ¿Qué estrategia distorsiona menos? ¿Qué le pasa a la desviación estándar?

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 3 · `deck` y `embark_town`, y un adelanto de inferencia
1. Propón y aplica un tratamiento para `deck` (77% faltante) y para `embark_town` (2 filas). Justifica.
2. **Adelanto de la sesión 6:** con `pd.crosstab` y `scipy.stats.chi2_contingency`, prueba si la falta de `age` es independiente de `class`. Interpreta el valor p con cuidado.
3. ¿La tasa de supervivencia es distinta entre quienes tienen y no tienen `age` registrada? ¿Qué implicaría eso para un análisis que simplemente borra esas filas?

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

### Parte B · Auditoría de `penguins` y dataset limpio para las siguientes sesiones

### 🟢 Ejercicio 9 · Aplica la misma auditoría a `penguins` (el original, no `sucio`)
1. Recorre la lista de verificación: tipos, duplicados, categorías (`value_counts`), rangos y faltantes por columna y por patrón.
2. Identifica las **filas sin medidas** y las filas con **`sex` faltante**. ¿La falta de `sex` se concentra en alguna especie o isla?
3. Decide y justifica: ¿qué haces con las filas sin medidas? ¿imputas `sex`?

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### Dataset limpio de referencia (`data/penguins_limpio.csv`)
Para que **todos empecemos la sesión 3 con los mismos datos**, la siguiente celda aplica las decisiones acordadas y guarda el resultado. Compárala con lo que hiciste en el ejercicio 9.

Decisiones: (1) `sex` → `Male`/`Female`; (2) eliminar las filas sin ninguna medida; (3) **no** imputar `sex`; (4) tipos: `category` para las categóricas e `Int64` para aleta y masa.

In [ ]:
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

penguins_limpio = (
    pinguinos
    .assign(sex=lambda d: d["sex"].str.title())                 # MALE -> Male
    .dropna(subset=medidas, how="all")                           # quita filas sin ninguna medida
    .astype({"species": "category", "island": "category", "sex": "category",
             "flipper_length_mm": "Int64", "body_mass_g": "Int64"})
    .reset_index(drop=True)
)

print("filas:", len(pinguinos), "->", len(penguins_limpio))
print("faltantes restantes:\n", penguins_limpio.isna().sum())

penguins_limpio.to_csv(DATA / "penguins_limpio.csv", index=False)
print("\nGuardado en", DATA / "penguins_limpio.csv")

### 🔴 Reto 4 · Impacto de las decisiones
Compara la masa media por especie y sexo en `pinguinos` (original) y en `penguins_limpio`. ¿Cambió algo? ¿Por qué era de esperarse ese resultado? Después, calcula cuánto cambiaría la masa media por especie si hubieras imputado `sex` con la **moda** de cada especie: ¿qué sexo "gana" artificialmente?

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---
## ✅ Cierre

**Hoy vimos:**
- La **lista de verificación de calidad** que usaremos en todo el curso.
- Duplicados exactos vs. por llave, y por qué un duplicado sin llave no siempre es un error (`titanic`).
- Inconsistencias: texto, categorías, códigos centinela, rangos imposibles y unidades.
- Mecanismos **MCAR / MAR / MNAR** y cómo perfilarlos.
- Tratamientos y su costo: eliminar sesga si no es MCAR; imputar con un valor único **encoge la varianza**; imputar por grupo es mejor si la causa está observada; guarda siempre el indicador.

**Práctica (sin calificación, en Classroom):** termina los ejercicios 🟢 y verifica que exista `data/penguins_limpio.csv`.

**Próxima sesión (mié 21 oct):** estadística descriptiva, forma de la distribución y outliers (`03_descriptiva_distribuciones.ipynb`), sobre `penguins_limpio`.